# INITIAL IMPORT

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import gymnasium as gym
from src.config import Configuration


CONFIG = Configuration(
    n_training_episodes = 1_000_000,
    learning_rate = 0.7,
    n_eval_episodes = 1000,
    gym_id = "Taxi-v4",
    max_steps = 1000,
    gamma = 0.90,
    max_epsilon = 1.0,
    min_epsilon = 0.05,
    decay_rate = 0.001,
)

# LOAD ENV

In [ ]:
env = gym.make(CONFIG.gym_id, render_mode="rgb_array")

state_space = env.observation_space.n
print("There are ", state_space, " possible states")
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

# DEFINE Q-LEARNING

In [ ]:
def initialize_q_table(state_space, action_space):
    """
    Is not a matrix, is an array and we can locate each game cell later with `current_row * ncols + current_col`
    """
    Qtable = np.zeros((state_space, action_space))
    return Qtable

def greedy_policy(Qtable, state):
    """Take the action with the highest value given a state
    """
    action = np.argmax(Qtable[state][:])
    return action

def epsilon_greedy_policy(Qtable, state, epsilon):
    """Take the action with the highest value given a state with a probability of 1-epsilon, otherwise take a random action
    """
    # Randomly generate a number between 0 and 1
    random_num = np.random.random()

    # if random_num > greater than epsilon --> exploitation
    if random_num > epsilon:
        action = greedy_policy(Qtable, state)
    # else --> exploration
    else:
        # np.random.randint(0, Qtable[state].size) # Take a random action
        action = env.action_space.sample() 

    return action

In [ ]:
from tqdm import tqdm

def train(CONFIG: Configuration, Qtable):
  env = gym.make(CONFIG.gym_id, render_mode="rgb_array")
  
  for episode in tqdm(range(CONFIG.n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = CONFIG.min_epsilon + (CONFIG.max_epsilon - CONFIG.min_epsilon)*np.exp(-CONFIG.decay_rate*episode)
    # Reset the environment
    state, info = env.reset(seed=CONFIG.seed + episode)
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(CONFIG.max_steps):
      # Choose the action At using epsilon greedy policy
      action = epsilon_greedy_policy(Qtable, state, epsilon)

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info = env.step(action)

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      old_Qsa = Qtable[state][action]
      future_reward = 0 if terminated else np.max(Qtable[new_state])
      Qtable[state][action] = old_Qsa + CONFIG.learning_rate * (reward + CONFIG.gamma * future_reward - old_Qsa)

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable

In [ ]:
def evaluate_agent(CONFIG: Configuration, Q):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for Taxi-v4)
  """
  env = gym.make(CONFIG.gym_id, render_mode="rgb_array")

  episode_rewards = []
  for episode in tqdm(range(CONFIG.n_eval_episodes)):
    if CONFIG.eval_seed:
      state, info = env.reset(seed=CONFIG.eval_seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(CONFIG.max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

In [ ]:
qt_taxi = initialize_q_table(state_space, action_space)
qt_taxi = train(CONFIG, qt_taxi)

In [ ]:
mean_reward, std_reward = evaluate_agent(CONFIG, qt_taxi)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
qt_taxi

# Record video

In [ ]:
import os
import random 
import imageio



def record_video(CONFIG: Configuration, Qtable):
  """
  Generate a replay video of the agent
  :param CONFIG: Configuration object
  :param Qtable: Qtable of our agent
  """
  images = []
  terminated = False
  truncated = False
  state, info = env.reset(seed=random.randint(0,500))
  img = env.render()
  images.append(img)
  while not (terminated or truncated):
    # Take the action (index) that have the maximum expected future reward given that state
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action) # We directly put next_state = state for recording logic
    img = env.render()
    images.append(img)
  
  path = os.path.join(CONFIG.VIDEO_PATH, f"{CONFIG.exp_description}_{CONFIG.gym_id}.gif")
  imageio.mimsave(path, [np.array(img) for i, img in enumerate(images)], fps=CONFIG.video_fps)

record_video(CONFIG, qt_taxi)